In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re

In [2]:
first_round = pd.read_csv("datos/classification first round/classification_final.csv")
second_round = pd.read_csv("datos/classification second_round/classification_secondRound_final.csv")

classifiers_first = pd.read_csv("datos/classification first round/classifiers_final.csv")
classifiers_second = pd.read_csv("datos/classification second_round/classifiers_secondRound_final.csv")

In [3]:
file_path = "_tempavisos.dta"

chunk_size = 10000
chunks = pd.read_stata(file_path, columns=['avisoid', 'avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo'], chunksize=chunk_size)

avisos = pd.DataFrame()

for chunk in chunks:
    avisos = pd.concat([avisos, chunk], ignore_index=True)

### Merging and counting tags accross rounds

In [4]:
# Define a mapping
classification_mapping = {
    'tot_remote': 'WFH',
    'temp_remote': 'WFH_partial',
    'semi_remote': 'WFH_partial',
    'semi_presence': 'WFH_partial',
    'not_clear_remote': 'WFH_partial',
    'not_remote': 'Not WFH'
}

# Apply the mapping
second_round['classification'] = second_round['classification'].replace(classification_mapping)

In [5]:
second_round['classification'].value_counts()

classification
WFH_partial    2736
WFH            1546
Not WFH         848
Name: count, dtype: int64

In [47]:
df_merged = pd.merge(first_round, second_round, on='ad_id', how='left')

In [48]:
# Step 3: Group by ad_id and collect both classification_x and classification_y as lists
grouped = df_merged.groupby('ad_id')[['classification_x', 'classification_y']].agg(list)

In [49]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (grouped['classification_y'].apply(lambda y: y[0] == y[1] == 'WFH'))
]
print(f"Number of ad_id with both classification_x and classification_y as 'WFH' twice: {len(filtered)}")

Number of ad_id with both classification_x and classification_y as 'WFH' twice: 643


In [50]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (grouped['classification_y'].apply(lambda y: y[0] == y[1] == 'WFH_partial'))
]
print(f"Number of ad_id with both classification_x 'WFH' and both classification_y as 'WFH_partial': {len(filtered)}")

Number of ad_id with both classification_x 'WFH' and both classification_y as 'WFH_partial': 1163


In [51]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'Not WFH'))
]
print(f"Number of ad_id with both classification_x 'Not WFH': {len(filtered)}")

Number of ad_id with both classification_x 'Not WFH': 5568


In [52]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (
        (grouped['classification_y'].apply(lambda y: (y[0] == 'WFH' and y[1] == 'WFH_partial'))) |
        (grouped['classification_y'].apply(lambda y: (y[0] == 'WFH_partial' and y[1] == 'WFH')))
    )
]

print(f"Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'WFH_partial': {len(filtered)}")

Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'WFH_partial': 196


In [53]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (
        (grouped['classification_y'].apply(lambda y: (y[0] == 'WFH' and y[1] == 'Not WFH'))) |
        (grouped['classification_y'].apply(lambda y: (y[0] == 'Not WFH' and y[1] == 'WFH')))
    )
]

print(f"Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'Not WFH': {len(filtered)}")

Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'Not WFH': 26


In [54]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (
        (grouped['classification_y'].apply(lambda y: (y[0] == 'WFH_partial' and y[1] == 'Not WFH'))) |
        (grouped['classification_y'].apply(lambda y: (y[0] == 'Not WFH' and y[1] == 'WFH_partial')))
    )
]

print(f"Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'Not WFH' or 'WFH_partial': {len(filtered)}")

Number of ad_id with both classification_x 'WFH' and at least one classification_y as 'Not WFH' or 'WFH_partial': 95


In [55]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
filtered = grouped[
    (grouped['classification_x'].apply(lambda x: x[0] == x[1] == 'WFH')) &
    (grouped['classification_y'].apply(lambda y: y[0] == y[1] == 'Not WFH'))
]
print(f"Number of ad_id with both classification_x 'WFH' and both classification_y as 'Not WFH': {len(filtered)}")

Number of ad_id with both classification_x 'WFH' and both classification_y as 'Not WFH': 309


In [56]:
# Step 4: Keep only rows where both entries in classification_x and classification_y are 'WFH'
# Group classifications by ad_id into a list
grouped = first_round.groupby('ad_id')['classification'].agg(list)

# Filter those where the two elements are not the same
diff_class = grouped[grouped.apply(lambda x: len(x) == 2 and x[0] != x[1])]

print(f"Number of ad_id with at least one classification_x 'WFH' and one 'Not WFH': {len(diff_class)}")

Number of ad_id with at least one classification_x 'WFH' and one 'Not WFH': 250


In [57]:
second_round['classifier_id'].nunique()

15

In [58]:
first_round['classifier_id'].nunique()

50

### Counting how many ads where coincident for each classification

In [6]:
# Step 1: Find ad_ids that appear exactly twice
ad_counts = second_round['ad_id'].value_counts()
ad_twice = ad_counts[ad_counts == 2].index

# Step 2: Filter original dataframe to only those ad_ids
twice_df = second_round[second_round['ad_id'].isin(ad_twice)]

# Step 3: Group by ad_id and aggregate classifications into a list
grouped = twice_df.groupby('ad_id')['classification'].apply(list)

# Step 4: Filter only those where both classifications are the same
same_class_ads = grouped[grouped.apply(lambda x: x[0] == x[1])]

# Step 5: Count the values of the identical classifications
from collections import Counter
result = Counter([x[0] for x in same_class_ads])

# Convert to DataFrame (optional)
import pandas as pd
result_df = pd.DataFrame(result.items(), columns=['classification', 'count'])


In [7]:
result_df

,classification,count
0,Not WFH,355
1,WFH,654
2,WFH_partial,1210


## Conservative approach

Retag the second round classifications

In [65]:
# Define a mapping
classification_mapping = {
    'tot_remote': 'WFH',
    'temp_remote': 'WFH_partial',
    'semi_remote': 'WFH_partial',
    'semi_presence': 'WFH_partial',
    'not_clear_remote': 'WFH_partial',
    'not_remote': 'Not WFH'
}

# Apply the mapping
second_round['classification'] = second_round['classification'].replace(classification_mapping)

In [66]:
# Step 1: Find ad_ids that appear exactly twice
ad_counts = second_round['ad_id'].value_counts()
ad_twice = ad_counts[ad_counts == 2].index

# Step 2: Filter original dataframe to only those ad_ids
twice_df = second_round[second_round['ad_id'].isin(ad_twice)]

# Step 3: Group by ad_id and aggregate classifications into a list
grouped = twice_df.groupby('ad_id')['classification'].apply(list)

# Step 4: Filter only those where both classifications are the same
same_class_ads = grouped[grouped.apply(lambda x: x[0] == x[1])]

# Step 5: Count the values of the identical classifications
from collections import Counter
result = Counter([x[0] for x in same_class_ads])

# Convert to DataFrame (optional)
import pandas as pd
result_df = pd.DataFrame(result.items(), columns=['classification', 'count'])


We take down all of the dissagreement within each round

In [67]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = first_round.groupby('ad_id')['classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
first_round_filtered = first_round[first_round['ad_id'].isin(valid_ad_ids)]

In [68]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = second_round.groupby('ad_id')['classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
second_round_filtered = second_round[second_round['ad_id'].isin(valid_ad_ids)]

Now we merge both rounds

In [69]:
classifications = pd.merge(first_round_filtered, second_round_filtered, on='ad_id', how='left')

We want to take all the dissagreement across rounds

In [70]:
# Make sure missing values are handled properly
classifications['classification_y'] = classifications['classification_y'].replace('', np.nan)

# Define the conditions
conditions = [
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH_partial') ),
    ( (classifications['classification_x'] == 'WFH_partial') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'Not WFH') & (classifications['classification_y'] == 'Not WFH') ),
    ( classifications['classification_y'].isna() )
]

# Define the results corresponding to each condition
choices = [
    'WFH',
    'WFH_partial',
    'WFH_partial',
    'Not WFH',
    classifications['classification_x']  # <== Take the value from x if y is missing
]

# Apply the logic
classifications['final_classification'] = np.select(
    conditions,
    choices,
    default=np.nan
)

Now we merge them with the avisos database

In [71]:
avisos['aviso'] = avisos[['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']].apply(
    lambda row: ' '.join(row.dropna().astype(str)), axis=1
)

In [72]:
# Function to clean HTML tags
def remove_html_tags(text):
    if pd.isnull(text):
        return ""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Clean and count words
avisos['aviso'] = avisos['aviso'].apply(remove_html_tags)

In [73]:
avisos = avisos[['avisoid', 'aviso']]

In [74]:
avisos = pd.merge(classifications, avisos, left_on='ad_id', right_on='avisoid', how='left')

In [75]:
avisos['final_classification'].value_counts()

final_classification
Not WFH        10870
WFH_partial     4440
WFH             3106
Name: count, dtype: int64

In [76]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = avisos.groupby('ad_id')['final_classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
avisos_filtered = avisos[avisos['ad_id'].isin(valid_ad_ids)]

In [77]:
avisos_filtered = avisos_filtered.drop_duplicates(subset=['aviso', 'final_classification'])

In [78]:
train_data = avisos_filtered[['aviso', 'final_classification']]

In [83]:
train_data.to_csv("datos/training/conservative_td3.csv", index=False, encoding='utf-8')

## Naive approach

Retag the second round classifications

In [4]:
# Define a mapping
classification_mapping = {
    'tot_remote': 'WFH',
    'temp_remote': 'WFH_partial',
    'semi_remote': 'WFH_partial',
    'semi_presence': 'WFH_partial',
    'not_clear_remote': 'WFH_partial',
    'not_remote': 'Not WFH'
}

# Apply the mapping
second_round['classification'] = second_round['classification'].replace(classification_mapping)

Now we merge both rounds

In [5]:
classifications = pd.merge(first_round, second_round, on='ad_id', how='left')

We want to take all the dissagreement across rounds

In [7]:
# Make sure missing values are handled properly
classifications['classification_y'] = classifications['classification_y'].replace('', np.nan)

# Define the conditions
conditions = [
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH_partial') ),
    ( (classifications['classification_x'] == 'WFH_partial') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'Not WFH') & (classifications['classification_y'] == 'Not WFH') ),
    ( classifications['classification_y'].isna() )
]

# Define the results corresponding to each condition
choices = [
    'WFH',
    'WFH_partial',
    'WFH_partial',
    'Not WFH',
    classifications['classification_x']  # <== Take the value from x if y is missing
]

# Apply the logic
classifications['final_classification'] = np.select(
    conditions,
    choices,
    default=np.nan
)

Now we merge them with the avisos database

In [8]:
avisos['aviso'] = avisos[['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']].apply(
    lambda row: ' '.join(row.dropna().astype(str)), axis=1
)

In [9]:
# Function to clean HTML tags
def remove_html_tags(text):
    if pd.isnull(text):
        return ""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Clean and count words
avisos['aviso'] = avisos['aviso'].apply(remove_html_tags)

In [10]:
avisos = avisos[['avisoid', 'aviso']]

In [11]:
avisos = pd.merge(classifications, avisos, left_on='ad_id', right_on='avisoid', how='left')

In [14]:
avisos['final_classification'].value_counts()

final_classification
Not WFH        11051
WFH_partial     5222
WFH             3023
Name: count, dtype: int64

In [ ]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = avisos.groupby('ad_id')['final_classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
avisos_filtered = avisos[avisos['ad_id'].isin(valid_ad_ids)]

In [25]:
avisos_filtered = avisos_filtered.drop_duplicates(subset=['aviso', 'final_classification'])

In [33]:
train_data = avisos_filtered[['avisoid', 'aviso', 'final_classification']]

In [37]:
train_data['final_classification'].value_counts()

final_classification
Not WFH        5482
WFH_partial    1279
WFH             673
Name: count, dtype: int64

In [36]:
train_data.to_csv("datos/training/naive_td1.csv", index=False, encoding='utf-8')

## Naive with consistent taggers

Retag the second round classifications

In [4]:
# Define a mapping
classification_mapping = {
    'tot_remote': 'WFH',
    'temp_remote': 'WFH_partial',
    'semi_remote': 'WFH_partial',
    'semi_presence': 'WFH_partial',
    'not_clear_remote': 'WFH_partial',
    'not_remote': 'Not WFH'
}

# Apply the mapping
second_round['classification'] = second_round['classification'].replace(classification_mapping)

We want to detect an indicator of disagreement by classifier

In [5]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = first_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_first = first_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier = df_first.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier['disagreement_rate'] = disagreement_by_classifier['total_disagreements'] / disagreement_by_classifier['total_ads']

In [6]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = second_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_second = second_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier_second = df_second.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier_second['disagreement_rate'] = disagreement_by_classifier_second['total_disagreements'] / disagreement_by_classifier_second['total_ads']

Filters for each round by classifier

In [7]:
disagreement_by_classifier = disagreement_by_classifier.loc[disagreement_by_classifier['disagreement_rate'] < 0.06]

In [8]:
disagreement_by_classifier_second = disagreement_by_classifier_second.loc[disagreement_by_classifier_second['disagreement_rate'] < 0.2]

In [9]:
first_round = first_round.loc[first_round['classifier_id'].isin(disagreement_by_classifier.index)]

In [10]:
second_round = second_round.loc[second_round['classifier_id'].isin(disagreement_by_classifier_second.index)]

Now we merge both rounds

In [11]:
classifications = pd.merge(first_round, second_round, on='ad_id', how='left')

In [12]:
len(classifications['classifier_id_x'].unique())

45

We want to take all the dissagreement across rounds

In [13]:
# Make sure missing values are handled properly
classifications['classification_y'] = classifications['classification_y'].replace('', np.nan)

# Define the conditions
conditions = [
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'WFH') & (classifications['classification_y'] == 'WFH_partial') ),
    ( (classifications['classification_x'] == 'WFH_partial') & (classifications['classification_y'] == 'WFH') ),
    ( (classifications['classification_x'] == 'Not WFH') & (classifications['classification_y'] == 'Not WFH') ),
    ( classifications['classification_y'].isna() )
]

# Define the results corresponding to each condition
choices = [
    'WFH',
    'WFH_partial',
    'WFH_partial',
    'Not WFH',
    classifications['classification_x']  # <== Take the value from x if y is missing
]

# Apply the logic
classifications['final_classification'] = np.select(
    conditions,
    choices,
    default=np.nan
)

Now we merge them with the avisos database

In [14]:
avisos['aviso'] = avisos[['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']].apply(
    lambda row: ' '.join(row.dropna().astype(str)), axis=1
)

In [15]:
# Function to clean HTML tags
def remove_html_tags(text):
    if pd.isnull(text):
        return ""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Clean and count words
avisos['aviso'] = avisos['aviso'].apply(remove_html_tags)

In [16]:
avisos = avisos[['avisoid', 'aviso']]

In [17]:
avisos = pd.merge(classifications, avisos, left_on='ad_id', right_on='avisoid', how='left')

In [19]:
avisos['final_classification'].value_counts()

final_classification
Not WFH        9962
WFH_partial    3623
WFH            2503
Name: count, dtype: int64

In [20]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = avisos.groupby('ad_id')['final_classification'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
avisos_filtered = avisos[avisos['ad_id'].isin(valid_ad_ids)]

In [21]:
avisos_filtered = avisos_filtered.drop_duplicates(subset=['aviso', 'final_classification'])

In [22]:
train_data = avisos_filtered[['avisoid', 'aviso', 'final_classification']]

In [24]:
train_data['final_classification'].value_counts()

final_classification
Not WFH        5093
WFH_partial    1086
WFH             785
Name: count, dtype: int64

In [25]:
train_data.to_csv("datos/training/naive_consistent_taggers_td1.csv", index=False, encoding='utf-8')